# Bitcoin Master Workflow
## Trustworthy Time-Series Foundation Model Evaluation — Finance

**Primary task:** daily Bitcoin Close forecasting  
**Authoritative protocol:** rolling one-step ahead  
**Authoritative test period:** 2023-08-12 through 2026-07-07  
**Authoritative artifact:** `results/validated_forecasts.csv`

This is a safe orchestration and analysis notebook. It complements—without replacing—the detailed phase notebooks.

## Table of Contents

1. Research Objective
2. Configuration and Safety Controls
3. Environment and Paths
4. Artifact Verification
5. Bitcoin Dataset Overview
6. Forecasting Protocol
7. Model Inventory
8. Optional Forecast Generation
9. Load Authoritative Forecasts
10. Point Accuracy
11. Forecast Visualization
12. Regime-Conditional Robustness
13. Temporal Stability
14. Uncertainty Calibration
15. Statistical Significance
16. Trustworthiness Evidence
17. Final Bitcoin Findings
18. Detailed Notebook Links

## Workflow

```text
Raw data (optional overview)
          |
          v
Frozen forecast vectors ---> schema/hash validation
          |                         |
          v                         v
Point metrics ----> robustness / stability / uncertainty / significance
          \_________________________/
                       |
                       v
          component evidence, then secondary composite summary
```

**Reproducibility notice:** Safe mode reads frozen artifacts only. It does not import TensorFlow, Chronos, TimesFM, or other model frameworks; train models; regenerate forecasts; or write authoritative results.

## 1. Research Objective

The objective is not merely to minimise MAE. The study evaluates Point Forecast Accuracy, Regime-Conditional Robustness, Temporal Stability, Uncertainty Calibration, Transparency and Auditability, Statistical Significance, and a secondary Exploratory Composite Trustworthiness Summary.

## 2. Configuration and Safety Controls

All destructive or expensive controls default to `False`. Changing `RUN_EXPENSIVE_MODELS` alone still does not overwrite anything: an explicit confirmation token is also required, and generation remains delegated to the detailed notebooks.

In [ ]:
RUN_EXPENSIVE_MODELS = False
OVERWRITE_AUTHORITATIVE_ARTIFACTS = False
VERIFY_ARTIFACTS = True
LOAD_RAW_DATA_OVERVIEW = True
EXPENSIVE_CONFIRMATION = ""  # must equal "I UNDERSTAND THIS MAY REGENERATE FORECASTS"

SAFETY = {
    "RUN_EXPENSIVE_MODELS": RUN_EXPENSIVE_MODELS,
    "OVERWRITE_AUTHORITATIVE_ARTIFACTS": OVERWRITE_AUTHORITATIVE_ARTIFACTS,
    "VERIFY_ARTIFACTS": VERIFY_ARTIFACTS,
    "LOAD_RAW_DATA_OVERVIEW": LOAD_RAW_DATA_OVERVIEW,
}
SAFETY

## 3. Environment and Paths

The next cell locates the repository from either the repository root or `notebooks/`, adds it to the import path, and defines portable paths.

In [ ]:
from pathlib import Path
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown


def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src").is_dir() and (candidate / "results").is_dir():
            return candidate
    raise FileNotFoundError("Repository root not found from the current working directory.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = ROOT / "figures"
NOTEBOOKS_DIR = ROOT / "notebooks"

from src.metrics import mae, rmse, mape, smape

pd.DataFrame({"Path": [ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR]}, index=["ROOT", "DATA_DIR", "RESULTS_DIR", "FIGURES_DIR"])

## 4. Artifact Verification

The lightweight repository verifier checks protected hashes, schemas, keys, row counts, and reproducible metrics. It never imports or invokes forecasting frameworks.

In [ ]:
verification_summary = "SKIPPED"
if VERIFY_ARTIFACTS:
    completed = subprocess.run(
        [sys.executable, str(ROOT / "src" / "verify_research_artifacts.py")],
        cwd=ROOT, capture_output=True, text=True, check=False,
    )
    lines = [line for line in completed.stdout.splitlines() if line.strip()]
    verification_summary = lines[-1] if lines else completed.stderr.strip()
    if completed.returncode != 0:
        raise RuntimeError(verification_summary)
display(pd.DataFrame([{"Artifact verification": verification_summary}]))

## 5. Bitcoin Dataset Overview

Only `Timestamp` and `Close` are read for this lightweight overview. If the raw source is unavailable, analysis continues from the frozen forecast vectors.

In [ ]:
bitcoin_path = DATA_DIR / "bitcoin" / "btcusd_1-min_data.csv"
if LOAD_RAW_DATA_OVERVIEW and bitcoin_path.is_file():
    minute = pd.read_csv(bitcoin_path, usecols=["Timestamp", "Close"])
    minute["Timestamp"] = pd.to_datetime(minute["Timestamp"], unit="s", utc=True)
    minute = minute.sort_values("Timestamp")
    daily_close = minute.set_index("Timestamp")["Close"].resample("D").last().dropna()
    overview = pd.DataFrame([{
        "Raw rows": len(minute), "Raw start": minute["Timestamp"].min(),
        "Raw end": minute["Timestamp"].max(), "Daily observations": len(daily_close),
        "Daily missing": int(daily_close.isna().sum()),
    }])
    display(overview)
    display(daily_close.describe().to_frame("Daily Close"))
    del minute
elif not bitcoin_path.is_file():
    display(Markdown("**Raw Bitcoin source data is not available locally. Continuing with frozen authoritative forecast artifacts.**"))
else:
    display(Markdown("Raw-data overview disabled; continuing with frozen authoritative forecast artifacts."))

## 6. Forecasting Protocol

The daily series uses an 80/20 chronological split: train ends **2023-08-11**; the 1,061-observation test begins **2023-08-12** and ends **2026-07-07**.

```text
history strictly before t ---> forecast(t) ---> reveal actual(t) ---> append for t+1
```

The target value and future test observations are never available before the forecast is recorded.

In [ ]:
protocol = pd.DataFrame([
    {"Split": "Train", "End": "2023-08-11", "Rule": "Historical 80%"},
    {"Split": "Test", "Start": "2023-08-12", "End": "2026-07-07", "Observations": 1061},
])
display(protocol)

## 7. Model Inventory

Only models with exact, aligned rolling one-step vectors enter the authoritative ranking.

In [ ]:
authoritative_models = pd.DataFrame([
    ("Naive", "Baseline", "Authoritative"),
    ("Persistence-Enhanced LSTM", "Deep Learning", "Authoritative"),
    ("Chronos-Bolt-Tiny", "Foundation Model", "Authoritative zero-shot"),
    ("TimesFM", "Foundation Model", "Authoritative zero-shot"),
], columns=["Model", "Class", "Status"])
exploratory_models = pd.DataFrame([
    ("7-Day Moving Average", "Deterministic benchmark"), ("ARIMA", "Protocol-limited"),
    ("SARIMA", "Protocol-limited"), ("Raw-price LSTM", "Exploratory"),
    ("Experimental LSTM", "Exploratory"), ("Transformers", "Exploratory failure studies"),
], columns=["Model", "Status"])
display(authoritative_models)
display(Markdown("**Exploratory or protocol-limited models**"))
display(exploratory_models)

## 8. OPTIONAL — EXPENSIVE / RESEARCH REGENERATION

Generation remains in [01_EDA.ipynb](01_EDA.ipynb), [03_Deep_Learning_LSTM.ipynb](03_Deep_Learning_LSTM.ipynb), [03b_LSTM_Improved.ipynb](03b_LSTM_Improved.ipynb), and [05_Foundation_Models.ipynb](05_Foundation_Models.ipynb). Safe `Run All` never runs them.

In [ ]:
if RUN_EXPENSIVE_MODELS:
    required = "I UNDERSTAND THIS MAY REGENERATE FORECASTS"
    if EXPENSIVE_CONFIRMATION != required:
        raise PermissionError("Expensive mode requires the exact confirmation token.")
    if OVERWRITE_AUTHORITATIVE_ARTIFACTS:
        raise PermissionError("Authoritative overwrite is intentionally unsupported in the master notebook.")
    display(Markdown("Use the linked generation notebooks after reviewing their model-specific controls."))
else:
    display(Markdown("**SAFE MODE:** expensive model generation skipped; no heavy model package was imported."))

## 9. Load Authoritative Forecasts

The frozen vector must contain exactly 1,061 complete, unique, chronologically sorted rows.

In [ ]:
forecast_path = RESULTS_DIR / "validated_forecasts.csv"
btc = pd.read_csv(forecast_path, parse_dates=["Timestamp"])
required = ["Timestamp", "Actual", "Naive", "Persistence_Enhanced_LSTM", "Chronos_Bolt_Tiny", "TimesFM"]
checks = {
    "1,061 rows": len(btc) == 1061,
    "required schema": btc.columns.tolist() == required,
    "unique timestamps": btc["Timestamp"].is_unique,
    "sorted timestamps": btc["Timestamp"].is_monotonic_increasing,
    "no missing values": not btc[required].isna().any().any(),
    "expected start": str(btc["Timestamp"].min().date()) == "2023-08-12",
    "expected end": str(btc["Timestamp"].max().date()) == "2026-07-07",
}
assert all(checks.values()), checks
display(pd.DataFrame({"Check": checks.keys(), "PASS": checks.values()}))

## 10. Point Accuracy

All metrics below are recomputed directly from the frozen vectors.

In [ ]:
labels = {
    "Naive": "Naive", "Persistence_Enhanced_LSTM": "Persistence-Enhanced LSTM",
    "Chronos_Bolt_Tiny": "Chronos-Bolt-Tiny", "TimesFM": "TimesFM",
}
rows = []
for column, label in labels.items():
    rows.append({"Model": label, "MAE": mae(btc.Actual, btc[column]), "RMSE": rmse(btc.Actual, btc[column]),
                 "MAPE": mape(btc.Actual, btc[column]), "sMAPE": smape(btc.Actual, btc[column])})
btc_metrics = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
btc_metrics.insert(0, "Rank", np.arange(1, len(btc_metrics) + 1))
display(btc_metrics.style.format({c: "{:.6f}" for c in ["MAE", "RMSE", "MAPE", "sMAPE"]}))

## 11. Forecast Visualization

A representative 180-day window keeps the comparison readable. No publication figure is written or overwritten.

In [ ]:
window = btc.tail(180).set_index("Timestamp")
ax = window[["Actual", "Naive", "Persistence_Enhanced_LSTM", "TimesFM", "Chronos_Bolt_Tiny"]].plot(figsize=(12, 5), linewidth=1.2)
ax.set(title="Bitcoin rolling one-step forecasts — final 180 test days", ylabel="BTC Close", xlabel="Date")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

## 12. Regime-Conditional Robustness

This is conditional subgroup performance—not adversarial robustness. The protected cross-domain foundation-model table provides relative Bitcoin regime-robustness scores; detailed regime metrics remain documented in the case study and source notebook.

In [ ]:
foundation = pd.read_csv(RESULTS_DIR / "cross_domain_foundation_model_comparison.csv")
btc_foundation = foundation[foundation["Domain"].eq("Bitcoin")].copy()
robustness = btc_foundation[["Model", "Relative_Robustness_Score"]].sort_values("Relative_Robustness_Score", ascending=False)
display(robustness)

## 13. Temporal Stability

Earlier, Middle, and Later segments are derived from the same frozen test vector. The protected table reports the relative summary historically labelled `Generalisation`; it is interpreted here as **Temporal Stability**, not broad generalisation.

In [ ]:
segment_labels = pd.qcut(np.arange(len(btc)), 3, labels=["Earlier", "Middle", "Later"])
segment_rows = []
for segment in ["Earlier", "Middle", "Later"]:
    part = btc[segment_labels == segment]
    for column, label in labels.items():
        segment_rows.append({"Segment": segment, "Model": label, "MAE": mae(part.Actual, part[column]), "RMSE": rmse(part.Actual, part[column])})
btc_segments = pd.DataFrame(segment_rows)
display(btc_segments.pivot(index="Model", columns="Segment", values="MAE").round(3))
display(btc_foundation[["Model", "Relative_Generalisation_Score"]].rename(columns={"Relative_Generalisation_Score": "Relative Temporal Stability Score"}))

## 14. Uncertainty Calibration

Chronos has lower absolute error from nominal 80% marginal coverage in this task. This is not a claim of universal calibration superiority; interval width and sharpness also matter.

In [ ]:
uncertainty = pd.read_csv(RESULTS_DIR / "cross_domain_uncertainty_comparison.csv")
btc_uncertainty = uncertainty[uncertainty["Domain"].eq("Bitcoin")][["Model", "Nominal_Coverage", "Empirical_Coverage", "Coverage_Error", "Average_Width", "Width_Units"]]
display(btc_uncertainty)

## 15. Statistical Significance

The protected cross-domain significance summary is loaded rather than recomputing or overwriting DM artifacts. The detailed implementation is in notebook 09.

In [ ]:
significance = pd.read_csv(RESULTS_DIR / "cross_domain_significance_summary.csv")
btc_significance = significance[significance["Domain"].eq("Bitcoin")].copy()
display(btc_significance[["Comparison", "Lower_Loss_Model", "p_value", "P_Value_Type", "Significant", "Practical_Interpretation"]])

## 16. Trustworthiness Evidence

Component-level evidence is primary. The exploratory composite is secondary, researcher-defined, comparison-set dependent, and not a universal trustworthiness metric.

In [ ]:
components = btc_foundation[["Model", "Point_Rank", "Relative_Robustness_Score", "Relative_Generalisation_Score", "Empirical_80_Coverage", "Coverage_Error"]].rename(columns={"Relative_Generalisation_Score": "Relative_Temporal_Stability_Score"})
composite = btc_foundation[["Model", "Penalised_Trust_Score"]]
display(Markdown("**Primary component evidence**")); display(components)
display(Markdown("**Secondary exploratory composite summary**")); display(composite)

## 17. Final Bitcoin Findings

- Naive is the strongest point forecaster.
- Persistence-Enhanced LSTM is the strongest supervised neural saved-vector model.
- TimesFM is the strongest zero-shot point foundation model.
- Chronos shows substantially better 80% marginal coverage behaviour than TimesFM.
- Complexity does not guarantee trustworthy performance.

## 18. Detailed Notebook Links

- [01 — EDA](01_EDA.ipynb)
- [02 — Classical Models](02_Classical_Models.ipynb)
- [03 — Deep Learning LSTM](03_Deep_Learning_LSTM.ipynb)
- [03b — Improved LSTM](03b_LSTM_Improved.ipynb)
- [04 — Transformers](04_Transformers.ipynb)
- [05 — Foundation Models](05_Foundation_Models.ipynb)
- [05 — Advanced Forecasting Models compatibility scaffold](05_Advanced_Forecasting_Models.ipynb)
- [06 — Trustworthiness](06_Trustworthiness.ipynb)
- [07 — Model Validation Audit](07_Model_Validation_Audit.ipynb)
- [08 — Naive Forecast Audit](08_Naive_Forecast_Audit.ipynb)
- [09 — Statistical Significance](09_Statistical_Significance_Test.ipynb)
- [18 — Cross-Domain Comparison](18_Cross_Domain_Comparison.ipynb)